# HDD 17 — Five-model shared masked Transformer training

Train a **single self-supervised temporal encoder** on the five provisional Notebook 13 HDD models. Each fit drive contributes at most one complete July 17–August 15 window (30 days, five SMART values). Drive-disjoint 10% of those fit drives are reserved for stopping; calibration and evaluation drives are never used for fitting. No failure flags are read. The model reconstructs randomly hidden SMART readings, and preprocessing is fit on training-drive windows only.

This notebook produces a reusable frozen checkpoint, preprocessing specification, training history and manifest. **It does not claim validated regimes or failure warnings.** Notebook 18 must score calibration/evaluation drives and assess anomalies before comparing to baselines. The Q3 failure summary was already inspected, so Q3 remains development evidence. Q4 and June remain closed.

Expect GPU training to be substantially faster. CPU training can take a while. Rerunning after an interruption resumes the last saved epoch with its optimizer state; a completed run reloads without retraining.

In [1]:
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import hashlib,json,random,re,zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
PILOT=COMPONENT_ROOT/'results'/RUN_ID/PILOT_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
MODEL_OUT=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
for folder in [OUT,MODEL_OUT]:folder.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def load_json(path):
    assert path.is_file(),f'Missing input: {path}'
    return json.loads(path.read_text())
audit=load_json(AUDIT/'multimodel_coverage_complete.json')
pilot=load_json(PILOT/'pilot_complete.json')
assert audit['analysis_complete'] and pilot['analysis_complete']
assert not audit['future_test_outcomes_opened'] and not pilot['future_test_outcomes_opened']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==audit['split_sha256']
reservation=load_json(reservation_path)
assert sha256(Q3_ZIP)==audit['settings']['archive_sha256']
models=list(reservation['models']);assert len(models)==5
model_index={m:i for i,m in enumerate(models)}
settings15=load_json(PILOT/'settings.json')
assert sha256(PILOT/'settings.json')==pilot['settings_sha256']
SALT=settings15['salt']
def group_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
fit_map={str(s):m for m,g in reservation['models'].items() for s in g['development'] if group_for(s)=='fit'}
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
assert set(fit_map).isdisjoint(reserved)
FEATURES=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert len(FEATURES)==5
SEED=42
SETTINGS={'seed':SEED,'features':FEATURES,'models':models,'window_start':'2021-07-17','window_end':'2021-08-15',
          'window_days':30,'one_window_per_fit_drive':True,
          'stopping_fraction_of_fit_drives':.10,'stopping_salt':'five-model-masked-transformer-stop-v1',
          'input_transform':'log1p then model-specific train-only median and IQR; constant IQR uses 1',
          'derived_input':'normalized adjacent-day differences, hidden when either source reading is hidden',
          'width':64,'heads':4,'layers':2,'feedforward':128,'dropout':.10,
          'mask_fraction':.15,'batch_size':256,'max_epochs':25,'patience':5,'learning_rate':0.0005,
          'minimum_delta':0.00001,
          'reservation_sha256':sha256(reservation_path),'archive_sha256':sha256(Q3_ZIP),
          'upstream_pilot_complete_sha256':sha256(PILOT/'pilot_complete.json')}
settings_path=OUT/'settings.json'
if settings_path.exists():assert load_json(settings_path)==SETTINGS,'Training protocol changed; use a new TRAIN_ID.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Declared fit drives:',len(fit_map),'| device:',DEVICE)
print('Failure flags and Q4 outcomes remain unopened in this notebook.')

Declared fit drives: 46545 | device: cpu
Failure flags and Q4 outcomes remain unopened in this notebook.


## 1. Extract the August 15 fitting windows

Only fit-drive IDs from Notebook 13/15 are read. The 30 days must be consecutive and all five values finite and nonnegative. A drive missing any day cannot enter training. The expected counts must match Notebook 15's independent August 15 fit snapshot. Outputs are cached with an input manifest.

In [2]:
window_path=OUT/'fit_windows.npz';window_manifest_path=OUT/'fit_windows_manifest.json'
window_identity=hashlib.sha256((sha256(settings_path)+sha256(reservation_path)+sha256(Q3_ZIP)).encode()).hexdigest()
if window_path.exists() and window_manifest_path.exists():
    wm=load_json(window_manifest_path)
    assert wm['identity']==window_identity and wm['sha256']==sha256(window_path)
    with np.load(window_path) as a:
        fit_raw=a['windows'];serials=a['serial_number'].astype(str);names=a['model'].astype(str)
        assert a['features'].tolist()==FEATURES
    print('Verified cached fit windows:',len(serials))
else:
    with zipfile.ZipFile(Q3_ZIP) as z:
        members={}
        for info in z.infolist():
            if info.is_dir() or '__MACOSX' in info.filename:continue
            match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
            if match and '2021-07-17'<=match.group(1)<='2021-08-15':
                assert match.group(1) not in members
                members[match.group(1)]=info.filename
    dates=pd.date_range('2021-07-17','2021-08-15')
    assert set(members)==set(dates.strftime('%Y-%m-%d'))
    history={};last={}
    with zipfile.ZipFile(Q3_ZIP) as z:
        for day,(date,member) in enumerate(sorted(members.items())):
            seen=set()
            with z.open(member) as f:
                for chunk in pd.read_csv(f,usecols=['date','serial_number']+FEATURES,
                                         dtype={'serial_number':str},chunksize=50000):
                    part=chunk.loc[chunk.serial_number.isin(fit_map)]
                    if part.empty:continue
                    assert part.date.astype(str).eq(date).all()
                    ids=part.serial_number.tolist()
                    assert len(ids)==len(set(ids)) and seen.isdisjoint(ids)
                    seen.update(ids)
                    values=part[FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float64)
                    for serial,v in zip(ids,values):
                        if not (np.isfinite(v).all() and (v>=0).all()):
                            history.pop(serial,None);last[serial]=day;continue
                        if last.get(serial,-2)!=day-1:history[serial]=deque(maxlen=30)
                        history.setdefault(serial,deque(maxlen=30)).append(v.astype(np.float32))
                        last[serial]=day
            if date.endswith(('-01','-15')):print('Read fit histories:',date)
    records=[(s,fit_map[s],np.stack(h)) for s,h in history.items() if len(h)==30 and last.get(s)==29]
    records.sort(key=lambda row:row[0])
    assert records,'No valid fit windows.'
    serials=np.array([s for s,m,w in records]);names=np.array([m for s,m,w in records]);fit_raw=np.stack([w for s,m,w in records])
    assert fit_raw.shape==(len(serials),30,5) and np.isfinite(fit_raw).all()
    assert len(set(serials))==len(serials) and set(serials).isdisjoint(reserved)
    np.savez_compressed(window_path,windows=fit_raw,serial_number=serials,model=names,features=np.array(FEATURES))
    window_manifest_path.write_text(json.dumps({'identity':window_identity,'sha256':sha256(window_path),'rows':len(serials)},indent=2))
expected=load_json(COMPONENT_ROOT/'models/trained'/RUN_ID/PILOT_ID/'if_specification.json')['thresholds']
for model in models:
    assert int((names==model).sum())==int(expected[model]['fit_drives']),f'August 15 fit count mismatch: {model}'
print('Validated August 15 drive counts for all five models.')

Read fit histories: 2021-08-01
Read fit histories: 2021-08-15
Validated August 15 drive counts for all five models.


## 2. Fix stopping drives and train-only preprocessing

The stopping split is a deterministic hash of the fit serial, made without labels or future data. Preprocessing medians/IQRs are computed **only on the training drives** for each model, never on stopping, calibration or evaluation drives. The model ID is explicit; a later unsupported hardware model must return an error rather than borrow another model's scaler.

In [3]:
def stop_flag(serial):
    h=hashlib.sha256((SETTINGS['stopping_salt']+'|'+str(serial)).encode()).hexdigest()[:16]
    return int(h,16)/16**16 < SETTINGS['stopping_fraction_of_fit_drives']
stop=np.array([stop_flag(s) for s in serials]);train=~stop
assert train.sum()>100 and stop.sum()>100
assert all((stop & (names==m)).sum()>10 and (train & (names==m)).sum()>100 for m in models)
assert set(serials[train]).isdisjoint(set(serials[stop]))
logged=np.log1p(fit_raw.astype(np.float64))
centers={};scales={}
for model in models:
    values=logged[train & (names==model)].reshape(-1,5)
    assert len(values)>100
    centers[model]=np.median(values,axis=0)
    q25,q75=np.quantile(values,[.25,.75],axis=0)
    spread=q75-q25
    scales[model]=np.where(spread>1e-8,spread,1.0)
scaled=np.empty_like(fit_raw,dtype=np.float32)
for model in models:
    select=names==model
    scaled[select]=((logged[select]-centers[model])/scales[model]).astype(np.float32)
assert np.isfinite(scaled).all()
model_ids=np.array([model_index[m] for m in names],dtype=np.int64)
preprocessing={'models':models,'features':FEATURES,'center':{m:centers[m].tolist() for m in models},
               'scale':{m:scales[m].tolist() for m in models},'fit_training_drives':int(train.sum()),
               'stopping_drives':int(stop.sum()),'fit_windows_sha256':sha256(window_path),
               'settings_sha256':sha256(settings_path)}
preprocess_path=MODEL_OUT/'preprocessing.json'
if preprocess_path.exists():assert load_json(preprocess_path)==preprocessing
else:preprocess_path.write_text(json.dumps(preprocessing,indent=2))
display(pd.DataFrame({'model':models,
                      'train_drives':[int((train&(names==m)).sum()) for m in models],
                      'stopping_drives':[int((stop&(names==m)).sum()) for m in models]}))

,model,train_drives,stopping_drives
0,TOSHIBA MG07ACA14TA,13825,1549
1,ST12000NM0008,8180,864
2,ST4000DM000,8130,871
3,ST8000NM0055,6173,691
4,HGST HMS5C4040BLE640,5448,641


## 3. Train one shared masked Transformer

The model sees a zero in every hidden input position **and an explicit observed mask**. MSE is calculated only on hidden targets. A learned model embedding accommodates model identity. The stopping set contains different drives. `last_epoch.pt` saves model and optimizer after every epoch; `best_model.pt` retains the minimum stopping loss. Neither contains failure labels.

In [4]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def forward(self,x,model_id,hide):
        observed=(~hide).to(x.dtype)
        masked=x.masked_fill(hide,0)
        delta=torch.zeros_like(x)
        delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        delta_visible=observed.clone()
        delta_visible[:,1:,:]=observed[:,1:,:]*observed[:,:-1,:]
        delta=delta*delta_visible
        h=self.input(torch.cat([masked,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.output(self.encoder(h))

def masked_loss(pred,x,hide):
    return ((pred-x).square()*hide).sum()/hide.sum().clamp_min(1)

def make_mask(shape,seed,device):
    generator=torch.Generator(device=device).manual_seed(seed)
    mask=torch.rand(shape,generator=generator,device=device)<SETTINGS['mask_fraction']
    # Every window contributes at least one hidden target.
    missing=~mask.flatten(1).any(1)
    if missing.any():mask[missing,0,0]=True
    return mask

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
network=MaskedTelemetryTransformer(len(models),SETTINGS['width'],SETTINGS['heads'],SETTINGS['layers'],
                                   SETTINGS['feedforward'],SETTINGS['dropout']).to(DEVICE)
optimizer=torch.optim.AdamW(network.parameters(),lr=SETTINGS['learning_rate'])
train_tensor=TensorDataset(torch.from_numpy(scaled[train]),torch.from_numpy(model_ids[train]))
stop_tensor=TensorDataset(torch.from_numpy(scaled[stop]),torch.from_numpy(model_ids[stop]))
state_path=MODEL_OUT/'last_epoch.pt';best_path=MODEL_OUT/'best_model.pt';history_path=OUT/'training_history.csv'
complete_path=OUT/'encoder_training_complete.json'
start=0;best=float('inf');stale=0;history=[]
if complete_path.exists():
    finished=load_json(complete_path)
    assert finished['analysis_complete'] and finished['settings_sha256']==sha256(settings_path)
    assert finished['preprocessing_sha256']==sha256(preprocess_path)
    assert sha256(best_path)==finished['best_checkpoint_sha256']
    print('Verified complete training; no retraining.')
else:
    if state_path.exists():
        checkpoint=torch.load(state_path,map_location=DEVICE,weights_only=False) # Own trusted local checkpoint.
        assert checkpoint['settings_sha256']==sha256(settings_path) and checkpoint['preprocessing_sha256']==sha256(preprocess_path)
        network.load_state_dict(checkpoint['model']);optimizer.load_state_dict(checkpoint['optimizer'])
        start=checkpoint['epoch']+1;best=checkpoint['best'];stale=checkpoint['stale'];history=checkpoint['history']
        assert best_path.exists(), 'Interrupted state has no best checkpoint.'
        print('Resuming after epoch:',start)
    for epoch in range(start,SETTINGS['max_epochs']):
        if stale>=SETTINGS['patience']:break
        network.train();train_sum=train_count=0
        generator=torch.Generator().manual_seed(SEED+epoch)
        loader=DataLoader(train_tensor,batch_size=SETTINGS['batch_size'],shuffle=True,generator=generator,num_workers=0)
        for batch_no,(x,m) in enumerate(loader):
            x=x.to(DEVICE);m=m.to(DEVICE)
            hide=make_mask(x.shape,SEED+epoch*100000+batch_no,DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=masked_loss(network(x,m,hide),x,hide)
            loss.backward();nn.utils.clip_grad_norm_(network.parameters(),1.0);optimizer.step()
            train_sum+=float(loss.detach())*int(hide.sum());train_count+=int(hide.sum())
        network.eval();stop_sum=stop_count=0
        with torch.no_grad():
            for batch_no,(x,m) in enumerate(DataLoader(stop_tensor,batch_size=SETTINGS['batch_size'],shuffle=False,num_workers=0)):
                x=x.to(DEVICE);m=m.to(DEVICE)
                hide=make_mask(x.shape,SEED+900000+batch_no,DEVICE)
                pred=network(x,m,hide)
                stop_sum+=float(((pred-x).square()*hide).sum());stop_count+=int(hide.sum())
        train_mse=train_sum/train_count;stop_mse=stop_sum/stop_count
        improved=stop_mse<best-SETTINGS['minimum_delta']
        if improved:
            best=stop_mse;stale=0
            torch.save({'state_dict':network.state_dict(),'epoch':epoch,'stop_mse':best,
                        'models':models,'settings_sha256':sha256(settings_path),
                        'preprocessing_sha256':sha256(preprocess_path)},best_path)
        else:stale+=1
        history.append({'epoch':epoch+1,'train_masked_mse':train_mse,'stopping_masked_mse':stop_mse,'best_so_far':best})
        pd.DataFrame(history).to_csv(history_path,index=False)
        torch.save({'model':network.state_dict(),'optimizer':optimizer.state_dict(),
                    'epoch':epoch,'best':best,'stale':stale,'history':history,
                    'settings_sha256':sha256(settings_path),'preprocessing_sha256':sha256(preprocess_path)},state_path)
        print(f'Epoch {epoch+1:02d} | train {train_mse:.6f} | stopping {stop_mse:.6f} | best {best:.6f}')
        if stale>=SETTINGS['patience']:break
    assert best_path.exists() and len(history)>0
    record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
            'settings_sha256':sha256(settings_path),'preprocessing_sha256':sha256(preprocess_path),
            'fit_windows_sha256':sha256(window_path),'best_checkpoint_sha256':sha256(best_path),
            'best_stopping_mse':best,'epochs_completed':len(history),
            'train_drives':int(train.sum()),'stopping_drives':int(stop.sum()),
            'failure_labels_used':False,'calibration_or_evaluation_drives_used':False,
            'future_test_outcomes_opened':False,'sealed_June_test_opened':False,
            'regime_discovery_validated':False,'failure_detection_validated':False}
    complete_path.write_text(json.dumps(record,indent=2))

display(pd.read_csv(history_path).tail())

Epoch 01 | train 0.281237 | stopping 0.080453 | best 0.080453
Epoch 02 | train 0.055938 | stopping 0.027865 | best 0.027865
Epoch 03 | train 0.036262 | stopping 0.024117 | best 0.024117
Epoch 04 | train 0.026880 | stopping 0.020229 | best 0.020229
Epoch 05 | train 0.024930 | stopping 0.017246 | best 0.017246
Epoch 06 | train 0.022620 | stopping 0.018033 | best 0.017246
Epoch 07 | train 0.020371 | stopping 0.016372 | best 0.016372
Epoch 08 | train 0.019804 | stopping 0.016732 | best 0.016372
Epoch 09 | train 0.018463 | stopping 0.017357 | best 0.016372
Epoch 10 | train 0.020531 | stopping 0.016561 | best 0.016372
Epoch 11 | train 0.019325 | stopping 0.015196 | best 0.015196
Epoch 12 | train 0.019912 | stopping 0.017601 | best 0.015196
Epoch 13 | train 0.016861 | stopping 0.014557 | best 0.014557
Epoch 14 | train 0.016907 | stopping 0.014980 | best 0.014557
Epoch 15 | train 0.016151 | stopping 0.014064 | best 0.014064
Epoch 16 | train 0.016678 | stopping 0.016045 | best 0.014064
Epoch 17

,epoch,train_masked_mse,stopping_masked_mse,best_so_far
20,21,0.014560,0.012200,0.012200
21,22,0.014223,0.013892,0.012200
22,23,0.014884,0.012024,0.012024
23,24,0.014530,0.012903,0.012024
24,25,0.014226,0.011659,0.011659


## 4. Verify the saved model and define the next scoring contract

A deterministic reload must reproduce predictions on representative stopping windows. The next notebook should produce model-specific calibrated scores for the untouched calibration/evaluation drive groups and then assess regimes, reconstruction sensitivity, event warnings and alert burden. No performance claim is made here.

In [5]:
finished=load_json(complete_path)
assert finished['analysis_complete'] and not finished['failure_labels_used']
assert sha256(best_path)==finished['best_checkpoint_sha256']
check=torch.load(best_path,map_location=DEVICE,weights_only=True)
assert check['settings_sha256']==sha256(settings_path)
network.load_state_dict(check['state_dict']);network.eval()
x=torch.from_numpy(scaled[stop][:min(8,int(stop.sum()))]).to(DEVICE)
m=torch.from_numpy(model_ids[stop][:len(x)]).to(DEVICE)
mask=make_mask(x.shape,12345,DEVICE)
with torch.no_grad():
    a=network(x,m,mask).cpu().numpy()
reloaded=MaskedTelemetryTransformer(len(models),SETTINGS['width'],SETTINGS['heads'],SETTINGS['layers'],
                                    SETTINGS['feedforward'],SETTINGS['dropout']).to(DEVICE)
reloaded.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=True)['state_dict']);reloaded.eval()
with torch.no_grad():
    b=reloaded(x,m,mask).cpu().numpy()
np.testing.assert_allclose(a,b,rtol=1e-6,atol=1e-6)
# Hidden readings cannot leak through the adjacent-day difference channels.
changed=x.clone();changed[mask]+=1.25
with torch.no_grad():leak_check=reloaded(changed,m,mask).cpu().numpy()
np.testing.assert_allclose(b,leak_check,rtol=1e-6,atol=1e-6)
assert sha256(reservation_path)==SETTINGS['reservation_sha256']
assert sha256(Q3_ZIP)==SETTINGS['archive_sha256']
print('NOTEBOOK 17 COMPLETE — five-model shared masked Transformer trained and reload verified.')
print('Next: calibration and frozen scoring on the reserved Q3 development groups; Q4 remains closed.')
print('Artifacts:',MODEL_OUT)

NOTEBOOK 17 COMPLETE — five-model shared masked Transformer trained and reload verified.
Next: calibration and frozen scoring on the reserved Q3 development groups; Q4 remains closed.
Artifacts: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\models\trained\single_model_q1q2_2021_v1\q3_five_model_masked_transformer_v1
